In [4]:
#importing libraries

In [1]:
import pandas as pd

In [2]:
# loading data
data=pd.read_excel('G:\chatbot\data\chatbot_data.xlsx')
data.head()

,Govrnate,City,StationName,Destination,Price
0,الدقهلية,المنصورة,الاتوبيس الجديد,6 اكتوبر,78.0
1,الدقهلية,المنصورة,الاتوبيس الجديد,مدينة بدر,72.5
2,الدقهلية,المنصورة,الاتوبيس الجديد,دمياط الجديدة,34.5
3,الدقهلية,المنصورة,الاتوبيس الجديد,عبود,59.0
4,الدقهلية,المنصورة,الاتوبيس الجديد,السلام,61.0


In [3]:
# checking data
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1284 entries, 0 to 1283
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Govrnate     1284 non-null   object 
 1   City         1284 non-null   object 
 2   StationName  1284 non-null   object 
 3   Destination  1284 non-null   object 
 4   Price        1284 non-null   float64
dtypes: float64(1), object(4)
memory usage: 50.3+ KB


In [4]:
#cleaning data
object_cols=data.select_dtypes('object').columns
data[object_cols]=data[object_cols].apply(lambda x:x.str.strip())
data.tail()

,Govrnate,City,StationName,Destination,Price
1279,القاهرة,حلوان,عين حلوان,الزقازيق,40.0
1280,القاهرة,حلوان,عين حلوان,الإسكندرية,125.0
1281,القاهرة,حلوان,عين حلوان,كفر الشيخ,60.0
1282,القاهرة,حلوان,عين حلوان,طنطا,53.0
1283,القاهرة,حلوان,عين حلوان,المحلة الكبرى,61.0


In [ ]:
# chunking and metadata
def chunking(df):
  chunks=[]
  metadata=[]
  for _,row in df.iterrows():
    chunk=f"من محافظة {row['Govrnate']} مدينة {row['City']} عبر محطة {row['StationName']} يمكن السفر إلى {row['Destination']} بسعر {row['Price']} جنيه"
    chunks.append(chunk)
    metadata.append({'governorate':row['Govrnate'],'city':row['City'],'station':row['StationName'],'destination':row['Destination'],'price':row['Price']})
  return chunks,metadata

chunks,metadata=chunking(data)

In [ ]:
# creating embeddings
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
# creating pinecone index
pinecone_api_key=os.getenv('PINECONE_DEFAULT_API_KEY')
pc=Pinecone(api_key=pinecone_api_key)

index_name='chatbot'
if index_name not in pc.list_indexes():
  pc.create_index(name=index_name,dimension=384,metric='cosine',serverless=ServerlessSpec(cloud='aws',region='us-east-1'))


In [ ]:
# uploading data to pinecone
vecstore=LC_pinecone.from_texts(chunks,embeddings,index_name=index_name,metadatas=metadata)

In [ ]:
# get gen ai response
gemini_api_key=os.getenv('GEMINI_API_KEY')
llm=ChatGoogleGenerativeAI(model='gemini-1.5-pro',temperature=0.3,api_key=gemini_api_key)

# creating retriever and prompt template
retriever=vecstore.as_retriever(search_kwargs={'k':3})
prompt_template='''أنت رفيق المساعد الذكى فى السفر . استخدم المعلومات التالية للرد على استفسارات المستخدم المتعلقة بالسفر من خلال الحافلات. إذا لم تكن المعلومات كافية للإجابة، فقل "عذرًا، لا أستطيع مساعدتك في هذا الاستفسار".
 المعلومات: {context} 
 السؤال: {question} 
 الرد:'''
PROMPT=PromptTemplate.from_template(prompt_template)

# creating retrieval qa chain
qa_chain=RetrievalQA.from_chain_type(llm=llm,retriever=retriever,return_source_documents=False,chain_type_kwargs={'prompt':PROMPT})

In [ ]:
# testing the chatbot
query='ما هي تكلفة السفر من القاهرة إلى الإسكندرية؟'
response=qa_chain.invoke({"query": query})
print(response)

In [ ]:
# data=pd.read_excel('/content/test.xlsx')
# data.head()
# data.info()
# object_cols=data.select_dtypes('object').columns
# data[object_cols]=data[object_cols].apply(lambda x:x.str.strip())
# data.tail()

# # chunking and metadata
# def chunking(df):
#   chunks=[]
#   metadata=[]
#   for _,row in df.iterrows():
#     chunk=f'من محافظة {row['Govrnate']} مدينة {row['City']} عبر محطة {row['StationName']} يمكن السفر إلى {row['Destination']} بسعر {row['Price']} جنيه'
#     chunks.append(chunk)
#     metadata.append({'governorate':row['Govrnate'],'city':row['City'],'station':row['StationName'],'destination':row['Destination'],'price':row['Price']})
#   return chunks,metadata


# chunks,metadata=chunking(data)
# # creating embeddings
# embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# pc=Pinecone(api_key=os.getenv('PINECONE_DEFAULT_API_KEY'))
# index=pc.Index(index_name='chatbot')
# vectors=[]
# for i, chunk in enumerate(chunks):
#   vector=embeddings.embed_query(chunk)
#   vectors.append((f'id_{i}',vector,metadata[i]))

# index.upsert(vectors=vectors)
# vs=Pinecone.from_texts(chunks,embeddings,index_name='chatbot',metadatas=metadata)